# Leakage-aware stacking and statistical meta-learners

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/meta-learning.ipynb)

Companion notebook for the ReliableML paper `docs/latex/meta-learning.tex`. It covers `MetaFeatureTransformer`, `MetaPipelineClassifier`, `PerfMetaSelect`, `CalibratedStackedClassifier`, the `StatisticalMeta*` classifiers and `halving_grid_search`.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Methods

**Out-of-fold (OOF) meta-features.** With $K$-fold stratified CV and base learners $f_1,\dots,f_M$, row $i$ in validation fold $k$ gets class-probability meta-features from models that never saw it:
$$z_{im}=\bigl(\hat p_{f_m^{(-k)}}(Y=c\mid x_i)\bigr)_{c\in\mathcal C},\qquad z_i=[z_{i1},\dots,z_{iM}].$$
The meta-estimator $g$ is fit on $\{(z_i,y_i)\}$. At prediction time each $f_m$ is refit on all training rows and $\hat p(x)=g([\hat p_{f_1}(x),\dots,\hat p_{f_M}(x)])$. Class columns are aligned across folds. Feeding $g$ *in-sample* base probabilities instead would let it trust overfit learners.

**Performance selection** (`PerfMetaSelect`): $\bar s_m=\frac1K\sum_k s(f_m^{(-k)},D_k)$ and $m^\star=\arg\max_m\bar s_m$ with the best learner refit on all data. The CV score $\bar s_{m^\star}$ is optimistically biased as an estimate of future performance because it is the maximum of several noisy scores - use an outer test set.

**Calibrated stacking** (`CalibratedStackedClassifier`): the meta-estimator is wrapped in `CalibratedClassifierCV` (sigmoid or isotonic).

**Statistical meta-learners** (binary only): LDA projections and KMeans cluster distances $d_k(x)$ are turned into features, e.g. $\psi_k(x)=\exp(-\gamma_kd_k(x)^2)$, optionally with trees and degree-2 interactions, then
$$P(Y=1\mid x)=\sigma(\beta_0+\beta^\top u(x)),\qquad \ell(\beta)=-\sum_i\{y_i\log p_i+(1-y_i)\log(1-p_i)\}+\lambda\lVert\beta_{-0}\rVert_2^2.$$
The NN variant averages $B$ Monte-Carlo-dropout passes, $\hat p=\frac1B\sum_b\sigma(f_{\theta,m_b}(x))$ - a stochastic summary, not a calibrated uncertainty interval.

`feature_importances` / `local_feature_importances` report raw-feature importance as the change in predicted probability when a feature is replaced by a baseline (a model-behaviour probe, not causality).

## 2. Data and an honest comparison protocol

All models are compared on an untouched outer test split. Breast cancer (binary) is the main dataset; a synthetic non-linear task shows where stacking can help.

In [ ]:
import time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine, make_classification
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ml_reliability import (
    CalibratedStackedClassifier, MetaFeatureTransformer, MetaPipelineClassifier, PerfMetaSelect,
    StatisticalMetaDTreeClassifier, StatisticalMetaLDAClassifier, StatisticalMetaNNClassifier,
    halving_grid_search,
)

warnings.filterwarnings("ignore")

def base_learners():
    return [
        ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
        ("forest", RandomForestClassifier(n_estimators=100, random_state=0)),
        ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
    ]

def compare(models, Xtr, ytr, Xte, yte):
    rows = []
    for name, model in models.items():
        start = time.time()
        model.fit(Xtr, ytr)
        proba = model.predict_proba(Xte)[:, 1]
        rows.append({"model": name, "AUC": roc_auc_score(yte, proba), "Brier": brier_score_loss(yte, proba),
                     "accuracy": accuracy_score(yte, model.predict(Xte)), "fit seconds": time.time() - start})
    return pd.DataFrame(rows).set_index("model")

bc = load_breast_cancer(as_frame=True)
X, y = bc.data, bc.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print("train", Xtr.shape, "test", Xte.shape)

In [ ]:
models = {f"base: {name}": est for name, est in base_learners()}
models["MetaPipeline (OOF, LR meta)"] = MetaPipelineClassifier(base_estimators=base_learners(), n_folds=5, random_state=0)
models["PerfMetaSelect"] = PerfMetaSelect(estimators=base_learners(), cv=5, scoring="roc_auc", random_state=0)
models["CalibratedStacked"] = CalibratedStackedClassifier(base_estimators=base_learners(), n_folds=5, random_state=0)
models["StatisticalMeta-LDA"] = StatisticalMetaLDAClassifier(n_clusters=4, n_mc_sims=5, cv=3, random_state=0)
models["StatisticalMeta-DTree"] = StatisticalMetaDTreeClassifier(n_clusters=6, n_mc_sims=5, cv=3, random_state=0)
models["StatisticalMeta-NN"] = StatisticalMetaNNClassifier(n_clusters=6, epochs=20, n_mc_sims=5, cv=3, random_state=0)
results = compare(models, Xtr, ytr, Xte, yte)
print(results.round(4).to_string())
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
results["AUC"].plot.barh(ax=axes[0], title="Test AUC"); results["Brier"].plot.barh(ax=axes[1], title="Test Brier (lower is better)")
axes[0].set_xlim(results["AUC"].min() - 0.01, 1.0); plt.tight_layout(); plt.show()
print("PerfMetaSelect selected:", models["PerfMetaSelect"].best_name_, {k: round(v, 4) for k, v in models["PerfMetaSelect"].cv_scores_.items()})

With ~170 test rows, differences of a few thousandths in AUC are within sampling noise; this table is a smoke test of the API and protocol, not a ranking of methods.

## 3. Why out-of-fold matters: a leakage demonstration

In [ ]:
# Noisy labels (20% flipped): a deep forest memorizes the training set, a linear model cannot.
Xn, yn = make_classification(n_samples=1500, n_features=12, n_informative=5, flip_y=0.2, class_sep=1.0, random_state=2)
Xntr, Xnte, yntr, ynte = train_test_split(Xn, yn, test_size=0.4, random_state=0, stratify=yn)
learners = [("forest", RandomForestClassifier(n_estimators=100, random_state=0)),
            ("logreg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)))]

Z_oof = MetaFeatureTransformer(estimators=learners, cv=5, random_state=0).fit_transform(Xntr, yntr)
fitted = [clone(estimator).fit(Xntr, yntr) for _, estimator in learners]
Z_in = np.hstack([m.predict_proba(Xntr) for m in fitted])
Z_test = np.hstack([m.predict_proba(Xnte) for m in fitted])
print("forest in-sample accuracy:", round(accuracy_score(yntr, fitted[0].predict(Xntr)), 3),
      "| forest test accuracy:", round(accuracy_score(ynte, fitted[0].predict(Xnte)), 3))
print("meta-feature columns: forest P(0), forest P(1), logreg P(0), logreg P(1)")
for name, Z in [("meta-learner on IN-SAMPLE features", Z_in), ("meta-learner on OOF features", Z_oof)]:
    meta = LogisticRegression(max_iter=1000).fit(Z, yntr)
    proba = meta.predict_proba(Z_test)[:, 1]
    print(f"{name:36s} coef={np.round(meta.coef_[0], 2)} test AUC={roc_auc_score(ynte, proba):.4f} Brier={brier_score_loss(ynte, proba):.4f}")

## 4. Inspect meta-features, importances and tuning

In [ ]:
mp = models["MetaPipeline (OOF, LR meta)"]
Z = mp.transform_meta_features(Xte)
print("meta-feature matrix for test rows:", Z.shape, "(M learners x 2 classes)")
imp = mp.feature_importances(Xte.iloc[:40], top_n=5)
print(pd.DataFrame(imp["features"]).round(4).to_string(index=False))

In [ ]:
search = halving_grid_search(
    MetaPipelineClassifier(base_estimators=base_learners(), n_folds=3, random_state=0),
    {"meta_classifier": [LogisticRegression(C=0.1, max_iter=2000), LogisticRegression(C=1.0, max_iter=2000),
                         LogisticRegression(C=10.0, max_iter=2000)]},
    cv=3, scoring="roc_auc", random_state=0, factor=2, min_resources="exhaust",
).fit(Xtr, ytr)
print("best meta-classifier:", search.best_params_["meta_classifier"], "| CV AUC", round(search.best_score_, 4))
print("outer test AUC of the tuned model:", round(roc_auc_score(yte, search.predict_proba(Xte)[:, 1]), 4))

## 5. Synthetic non-linear task and a multiclass example

In [ ]:
Xs, ys = make_classification(n_samples=1500, n_features=12, n_informative=6, n_redundant=2, n_clusters_per_class=3,
                             flip_y=0.05, class_sep=0.8, random_state=1)
Xs = pd.DataFrame(Xs, columns=[f"f{i}" for i in range(12)])
Xstr, Xste, ystr, yste = train_test_split(Xs, ys, test_size=0.3, random_state=0, stratify=ys)
synth = {"logreg": base_learners()[0][1], "forest": base_learners()[1][1],
         "MetaPipeline": MetaPipelineClassifier(base_estimators=base_learners(), n_folds=5, random_state=0),
         "StatisticalMeta-LDA": StatisticalMetaLDAClassifier(n_clusters=6, n_mc_sims=5, cv=3, random_state=0)}
print(compare(synth, Xstr, ystr, Xste, yste).round(4).to_string())

wine = load_wine(as_frame=True)
Xwtr, Xwte, ywtr, ywte = train_test_split(wine.data, wine.target, test_size=0.3, random_state=0, stratify=wine.target)
selector = PerfMetaSelect(estimators=base_learners(), cv=5, scoring="accuracy", random_state=0).fit(Xwtr, ywtr)
stack = MetaPipelineClassifier(base_estimators=base_learners(), n_folds=5, random_state=0).fit(Xwtr, ywtr)
print("wine (3 classes) | PerfMetaSelect picked", selector.best_name_, "acc", round(accuracy_score(ywte, selector.predict(Xwte)), 4),
      "| MetaPipeline acc", round(accuracy_score(ywte, stack.predict(Xwte)), 4))

## Interpretation and limitations

**How to read the output.** Compare methods only on data none of them touched. OOF meta-features keep the meta-learner honest about how reliable each base learner is on unseen rows; the leakage cell shows how in-sample probabilities of a flexible learner look nearly perfect and mislead the meta-learner. Selection scores and tuning scores are optimistic; calibrated stacking changes probability quality (Brier), not necessarily ranking.

**Limitations.** Stacking is not guaranteed to beat the best base learner, particularly on small data (a few test rows decide the ranking); extra computation scales with folds x learners; the statistical meta-learners are binary only and have many hyper-parameters; Monte-Carlo averaging is not an uncertainty interval; OOF folds assume exchangeable rows (use grouped or temporal splits for clustered or time-ordered data); feature importances probe model behaviour under baseline replacement and are not causal.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.